<a href="https://colab.research.google.com/github/alicienty/HW2/blob/main/Lukyanchikova_w2v_hw_ipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

В этом практикуме мы рассмотрим работу с библиотекой **Gensim** для работы с векторными представлениями текста

Мы рассмотрим
- **Word2Vec** - векторные представления слов
- **FastText** - улучшенные представления с учетом морфологии  
- **Doc2Vec** - векторные представления документов


In [1]:
!pip install gensim

import gensim
import gensim.downloader as api
from gensim.models import Word2Vec, FastText, Doc2Vec
from gensim.models.doc2vec import TaggedDocument
import numpy as np

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 21.8 MB/s eta 0:00:00


## Часть 1: Word2Vec

### Что такое Word2Vec?

Word2Vec преобразует слова в векторы чисел так, что семантически похожие слова оказываются близко в векторном пространстве.

**Два основных алгоритма:**
- **CBOW** - предсказывает слово по контексту
- **Skip-gram** - предсказывает контекст по слову

**Загрузка предобученной модели**

In [27]:
w2v_model = api.load('glove-wiki-gigaword-100')

print(f"Размер словаря: {len(w2v_model.key_to_index)}")
print(f"Размерность векторов: {w2v_model.vector_size}")

Размер словаря: 400000
Размерность векторов: 100


Найдите документацию `gensim`: какие датасеты кроме `glove-wiki-gigaword-100` доступны в библиотеке?

Выберите 3 датасета и кратко опишите их (источник данных, примерный объем, зачем такой датасет может использоваться)

Все доступные датасеты:


In [19]:
info_datasets = api.info()
for n in info_datasets['corpora'].keys():
  print(n)
for n in info_datasets['models'].keys():
  print(n)

semeval-2016-2017-task3-subtaskBC
semeval-2016-2017-task3-subtaskA-unannotated
patent-2017
quora-duplicate-questions
wiki-english-20171001
text8
fake-news
20-newsgroups
__testing_matrix-synopsis
__testing_multipart-matrix-synopsis
fasttext-wiki-news-subwords-300
conceptnet-numberbatch-17-06-300
word2vec-ruscorpora-300
word2vec-google-news-300
glove-wiki-gigaword-50
glove-wiki-gigaword-100
glove-wiki-gigaword-200
glove-wiki-gigaword-300
glove-twitter-25
glove-twitter-50
glove-twitter-100
glove-twitter-200
__testing_word2vec-matrix-synopsis


In [16]:
dataset_info = api.info("fake-news")
dataset_info

{'num_records': 12999,
 'record_format': 'dict',
 'file_size': 20102776,
 'reader_code': 'https://github.com/RaRe-Technologies/gensim-data/releases/download/fake-news/__init__.py',
 'license': 'https://creativecommons.org/publicdomain/zero/1.0/',
 'fields': {'crawled': 'date the story was archived',
  'ord_in_thread': '',
  'published': 'date published',
  'participants_count': 'number of participants',
  'shares': 'number of Facebook shares',
  'replies_count': 'number of replies',
  'main_img_url': 'image from story',
  'spam_score': 'data from webhose.io',
  'uuid': 'unique identifier',
  'language': 'data from webhose.io',
  'title': 'title of story',
  'country': 'data from webhose.io',
  'domain_rank': 'data from webhose.io',
  'author': 'author of story',
  'comments': 'number of Facebook comments',
  'site_url': 'site URL from BS detector',
  'text': 'text of story',
  'thread_title': '',
  'type': 'type of website (label from BS detector)',
  'likes': 'number of Facebook likes

fake-news: датасет фейковых новостей

**Источник данных**: новостные агрегаторы (244 сайта), собрано через API webhose.io

**Примерный объем**: 12999 новостей

**Для чего может использоваться**: датасет призван упростить задачу распознания фейковых новостей и остановить их вирусное распространение

In [25]:
dataset_info = api.info("glove-twitter-200")
dataset_info

{'num_records': 1193514,
 'file_size': 795373100,
 'base_dataset': 'Twitter (2B tweets, 27B tokens, 1.2M vocab, uncased)',
 'reader_code': 'https://github.com/RaRe-Technologies/gensim-data/releases/download/glove-twitter-200/__init__.py',
 'license': 'http://opendatacommons.org/licenses/pddl/',
 'parameters': {'dimension': 200},
 'description': 'Pre-trained vectors based on 2B tweets, 27B tokens, 1.2M vocab, uncased (https://nlp.stanford.edu/projects/glove/).',
 'preprocessing': 'Converted to w2v format with `python -m gensim.scripts.glove2word2vec -i <fname> -o glove-twitter-200.txt`.',
 'read_more': ['https://nlp.stanford.edu/projects/glove/',
  'https://nlp.stanford.edu/pubs/glove.pdf'],
 'checksum': 'e52e8392d1860b95d5308a525817d8f9',
 'file_name': 'glove-twitter-200.gz',
 'parts': 1}

glove-twitter-200: модель предобученная на датасете из твиттов

**Источник данных**: Твиттер

**Примерный объем**: 27 миллиардов токенов

**Для чего может использоваться**: исследование современного устно-письменного языка (в интернет-пространстве), политического дискурса и тд.; обучение генеративной нейросети для, например, бота (в твиттере люди отвечают друг другу на твитты) или для колонки (главная лингвистка Алисы говорит, что ее обучали как раз на русском твиттере, поэтому она такая токсичная раньше была:)

In [21]:
dataset_info = api.info("word2vec-ruscorpora-300")
dataset_info

{'num_records': 184973,
 'file_size': 208427381,
 'base_dataset': 'Russian National Corpus (about 250M words)',
 'reader_code': 'https://github.com/RaRe-Technologies/gensim-data/releases/download/word2vec-ruscorpora-300/__init__.py',
 'license': 'https://creativecommons.org/licenses/by/4.0/deed.en',
 'parameters': {'dimension': 300, 'window_size': 10},
 'description': 'Word2vec Continuous Skipgram vectors trained on full Russian National Corpus (about 250M words). The model contains 185K words.',
 'preprocessing': 'The corpus was lemmatized and tagged with Universal PoS',
 'read_more': ['https://www.academia.edu/24306935/WebVectors_a_Toolkit_for_Building_Web_Interfaces_for_Vector_Semantic_Models',
  'http://rusvectores.org/en/',
  'https://github.com/RaRe-Technologies/gensim-data/issues/3'],
 'checksum': '9bdebdc8ae6d17d20839dd9b5af10bc4',
 'file_name': 'word2vec-ruscorpora-300.gz',
 'parts': 1}

word2vec-ruscorpora-300: модель, обученная на НКРЯ

**Источник данных**: НКРЯ

**Примерный объем**: около 250 миллионов слов

**Для чего может использоваться**: лингвистические исследования, дискурсивные исследования; обучение генеративной нейросети на русском языке; обучение генеративной нейросети для создания текстов конкретного жанра или рода (например, в НКРЯ большой объем поэтического подкорпуса, если это можно достать из данных, можно попробовать научить модель сочинять стихи)

**Базовые операции с векторами**

In [29]:
# Получаем вектор слова
vector = w2v_model['computer']
print(f"Вектор слова 'computer': {vector[:5]}...")  # Показываем первые 5 чисел

# Вычисляем схожесть между словами
similarity = w2v_model.similarity('computer', 'laptop')
print(f"Схожесть 'computer' и 'laptop': {similarity:.4f}")

Вектор слова 'computer': [-0.16298   0.30141   0.57978   0.066548  0.45835 ]...
Схожесть 'computer' и 'laptop': 0.7024


**Поиск похожих слов**

In [30]:
# Находим похожие слова
similar_words = w2v_model.most_similar('python', topn=5)
print("Слова, похожие на 'python':")
for word, score in similar_words:
    print(f"  {word}: {score:.4f}")

Слова, похожие на 'python':
  monty: 0.6886
  php: 0.5865
  perl: 0.5784
  cleese: 0.5447
  flipper: 0.5113


*Ваш ответ здесь*

**Задание**

1. Загрузите любой датасет из gensim на ваш выбор

In [31]:
dataset = api.load("fake-news")

[==================================================] 100.0% 19.2/19.2MB downloaded


2. Напишите функцию, которая принимает на вход любое слово и вовращает 10 наиболее близких по вектору слов

In [36]:
def most_similar_words(word):
  vector = w2v_model[str(word)]
  similar_words = w2v_model.most_similar(str(word), topn=10)
  return similar_words

In [42]:
print(f"Слова, похожие на doggy:")
similar_words = most_similar_words("doggy")
for word, score in similar_words:
    print(f"{word}: {score:.4f}")

Слова, похожие на doggy:
dogg: 0.8085
snoop: 0.7900
dre: 0.6079
doggie: 0.5774
pimp: 0.5530
doggystyle: 0.5427
2pac: 0.5356
shakur: 0.4879
kurupt: 0.4873
gangsta: 0.4851


3. Обучите модель Word2Vec на тестовом датасете из ячейки ниже

Примените следующие настройки:

- размер вектора: 50
- размер окна: 3
- минимальная частота слова: 1
- потоков: 2
- использовать skip-gram

In [43]:
cooking_sentences = [
    ['варить', 'суп', 'овощи', 'морковь', 'картофель'],
    ['жарить', 'курица', 'сковорода', 'масло', 'специи'],
    ['печь', 'хлеб', 'мука', 'дрожжи', 'духовка'],
    ['резать', 'овощи', 'салат', 'помидоры', 'огурцы'],
    ['смешивать', 'ингредиенты', 'тесто', 'яйца', 'молоко'],
    ['варить', 'паста', 'вода', 'соль', 'соус'],
    ['гриль', 'мясо', 'овощи', 'уголь', 'барбекю'],
    ['тушить', 'говядина', 'горшок', 'вино', 'травы'],
    ['запекать', 'рыба', 'лимон', 'духовка', 'фольга'],
    ['готовить', 'завтрак', 'яичница', 'бекон', 'тост'],
    ['месить', 'тесто', 'пирог', 'начинка', 'яблоки'],
    ['кипятить', 'вода', 'чай', 'кофе', 'чашка'],
    ['мариновать', 'мясо', 'соус', 'специи', 'холодильник'],
    ['взбивать', 'сливки', 'сахар', 'десерт', 'торт'],
    ['парить', 'овощи', 'здоровое', 'питание', 'брокколи']
]

In [55]:
model = Word2Vec(sentences=cooking_sentences, vector_size=50, window=3, min_count=1, workers=2, sg=1)

In [56]:
print(f"Слова в словаре: {list(model.wv.key_to_index.keys())[:10]}...")

Слова в словаре: ['овощи', 'мясо', 'соус', 'вода', 'тесто', 'духовка', 'специи', 'варить', 'брокколи', 'питание']...


4. Проверьте модель

In [57]:
# Проверяем похожие слова в кулинарной тематике
try:
    similar = model.wv.most_similar('варить', topn=5)
    print("Слова, похожие на 'варить':")
    for word, score in similar:
        print(f"  {word}: {score:.4f}")
except KeyError:
    print("Слово 'варить' не найдено в словаре")

Слова, похожие на 'варить':
  вино: 0.2398
  ингредиенты: 0.2172
  хлеб: 0.1938
  брокколи: 0.1846
  кипятить: 0.1711


In [62]:
# Найдите слова, похожие на "духовка"
### ваш код здесь ###
try:
    word_1 = 'духовка'
    similar = model.wv.most_similar(word_1, topn=5)
    print(f"Слова, похожие на '{word_1}':")
    for word, score in similar:
        print(f"  {word}: {score:.4f}")
except KeyError:
    print(f"Слово {word_1} не найдено в словаре")

# Найдите слова, похожие на "овощи"
### ваш код здесь ###
try:
    word_1 = 'овощи'
    similar = model.wv.most_similar(word_1, topn=5)
    print(f"Слова, похожие на '{word_1}':")
    for word, score in similar:
        print(f"  {word}: {score:.4f}")
except KeyError:
    print(f"Слово {word_1} не найдено в словаре")

Слова, похожие на 'духовка':
  ингредиенты: 0.3199
  десерт: 0.3064
  холодильник: 0.2705
  питание: 0.2243
  пирог: 0.2142
Слова, похожие на 'овощи':
  мариновать: 0.2716
  хлеб: 0.2691
  гриль: 0.2546
  фольга: 0.2409
  сахар: 0.2108


## Часть 2: FastText

FastText улучшает Word2Vec, рассматривая слова как наборы символов (n-грамм). Это позволяет работать с редкими словами и опечатками

5. Обучите FastText на корпусе текстов из пункта 3. Используйте код ниже

In [98]:
ft_model = FastText(
    sentences=cooking_sentences,
    vector_size=50,
    window=3,
    min_count=1,
    workers=2
)

6. Найдите слова, похожие на "варить", "духовка" и "овощи" с помощью обученной модели. Используйте код из пункта 4

In [65]:
words = ['варить', 'духовка', 'овощи']

for w in words:
  try:
    similar = ft_model.wv.most_similar(w, topn=5)
    print(f"Слова, похожие на '{w}':")
    for word, score in similar:
      print(f"  {word}: {score:.4f}")
  except KeyError:
    print(f"Слово {w} не найдено в словаре")

Слова, похожие на 'варить':
  жарить: 0.5353
  парить: 0.4805
  месить: 0.3541
  тушить: 0.3405
  специи: 0.2622
Слова, похожие на 'духовка':
  взбивать: 0.4565
  лимон: 0.3561
  салат: 0.3050
  курица: 0.3041
  тост: 0.2944
Слова, похожие на 'овощи':
  жарить: 0.2960
  фольга: 0.2574
  морковь: 0.2297
  соус: 0.2172
  торт: 0.2094


7. Сравните модели

Дана функция для сравнения Word2Vec и FastText

Придумайте 3 слова с опечатками и проверьте, найдет ли их FastText и Word2Vec

In [66]:
def compare_models(word):
    """Сравнивает представления слова в разных моделях"""
    print(f"\nСравнение для слова: '{word}'")

    # Word2Vec
    try:
        w2v_similar = model.wv.most_similar(word, topn=2)
        print(f"  Word2Vec: {[w for w, _ in w2v_similar]}")
    except KeyError:
        print(f"  Word2Vec: слово не найдено")

    # FastText
    try:
        ft_similar = ft_model.wv.most_similar(word, topn=2)
        print(f"  FastText: {[w for w, _ in ft_similar]}")
    except KeyError:
        print(f"  FastText: слово не найдено")

# Сравниваем для разных слов
compare_models('learning')
compare_models('neural')


Сравнение для слова: 'learning'
  Word2Vec: слово не найдено
  FastText: ['духовка', 'пирог']

Сравнение для слова: 'neural'
  Word2Vec: слово не найдено
  FastText: ['мука', 'травы']


In [73]:
# Сравниваем для слов без ошибок
compare_models('собачка')
compare_models('Дезоксирибонуклеопротеидфосфорилирование')


Сравнение для слова: 'собачка'
  Word2Vec: слово не найдено
  FastText: ['питание', 'масло']

Сравнение для слова: 'Дезоксирибонуклеопротеидфосфорилирование'
  Word2Vec: слово не найдено
  FastText: ['специи', 'холодильник']


In [74]:
# Сравниваем для слов с ошибками
compare_models('сомбачка')
compare_models('амбмивалентность')


Сравнение для слова: 'сомбачка'
  Word2Vec: слово не найдено
  FastText: ['питание', 'тушить']

Сравнение для слова: 'амбмивалентность'
  Word2Vec: слово не найдено
  FastText: ['масло', 'печь']


## Часть 3: Doc2Vec

Doc2Vec расширяет Word2Vec для создания векторных представлений целых документов (предложений, абзацев, статей)

In [75]:
# Создаем размеченные документы
documents = [
    "machine learning is interesting",
    "deep learning uses neural networks",
    "python programming for data science",
    "artificial intelligence is amazing",
    "computer vision processes images"
]

# Преобразуем в формат TaggedDocument
tagged_docs = []
for i, doc in enumerate(documents):
    tokens = doc.split()
    tagged_doc = TaggedDocument(words=tokens, tags=[f"doc_{i}"])
    tagged_docs.append(tagged_doc)

print("Размеченные документы:")
for doc in tagged_docs[:3]:
    print(f"  Слова: {doc.words}")
    print(f"  Тег: {doc.tags}")

Размеченные документы:
  Слова: ['machine', 'learning', 'is', 'interesting']
  Тег: ['doc_0']
  Слова: ['deep', 'learning', 'uses', 'neural', 'networks']
  Тег: ['doc_1']
  Слова: ['python', 'programming', 'for', 'data', 'science']
  Тег: ['doc_2']


In [76]:
# Обучаем Doc2Vec
doc_model = Doc2Vec(
    documents=tagged_docs,
    vector_size=50,
    window=3,
    min_count=1,
    workers=2,
    epochs=20
)

print("Doc2Vec модель обучена!")
print(f"Количество документов: {len(doc_model.dv.key_to_index)}")

Doc2Vec модель обучена!
Количество документов: 5


In [77]:
# Получаем вектор документа
doc_vector = doc_model.dv["doc_0"]
print(f"Вектор документа doc_0: {doc_vector[:5]}...")

# Находим похожие документы
similar_docs = doc_model.dv.most_similar("doc_0", topn=2)
print("\nДокументы, похожие на doc_0:")
for doc_tag, similarity in similar_docs:
    doc_id = int(doc_tag.split('_')[1])
    print(f"  {doc_tag}: {similarity:.4f}")
    print(f"    Текст: {documents[doc_id]}")

Вектор документа doc_0: [-0.01057    -0.01198188 -0.01982618  0.01710627  0.00710373]...

Документы, похожие на doc_0:
  doc_1: 0.2735
    Текст: deep learning uses neural networks
  doc_2: 0.1275
    Текст: python programming for data science


In [78]:
# Сравниваем схожесть документов
def compare_documents(doc1_id, doc2_id):
    similarity = doc_model.dv.similarity(f"doc_{doc1_id}", f"doc_{doc2_id}")
    print(f"Схожесть doc_{doc1_id} и doc_{doc2_id}: {similarity:.4f}")
    print(f"  doc_{doc1_id}: {documents[doc1_id]}")
    print(f"  doc_{doc2_id}: {documents[doc2_id]}")

compare_documents(0, 1)  # machine learning vs deep learning
compare_documents(0, 3)  # machine learning vs AI

Схожесть doc_0 и doc_1: 0.2735
  doc_0: machine learning is interesting
  doc_1: deep learning uses neural networks
Схожесть doc_0 и doc_3: -0.0822
  doc_0: machine learning is interesting
  doc_3: artificial intelligence is amazing


8. Сравните схожесть doc_2 и doc_4

In [79]:
compare_documents(2, 4)

Схожесть doc_2 и doc_4: -0.0362
  doc_2: python programming for data science
  doc_4: computer vision processes images


9. Найдите самый похожий документ на doc_1

In [95]:
def find_similarity_the_most():
  x = input(f"Введите номер документа, для которого ищете похожий: ")
  most_similar_doc = doc_model.dv.most_similar("doc_"+x, topn=1)
  print(f"\nДокументы, похожие на doc_{x}:")
  for doc_tag, similarity in similar_docs:
    doc_id = int(doc_tag.split('_')[1])
    print(f"  {doc_tag}: {similarity:.4f}")
    print(f"    Текст: {documents[doc_id]}")

In [96]:
find_similarity_the_most()

Введите номер документа, для которого ищете похожий: 1

Документы, похожие на doc_1:
  doc_1: 0.2735
    Текст: deep learning uses neural networks
  doc_2: 0.1275
    Текст: python programming for data science


10. Выберите любую из трёх моделей. Обучите модели с разной размерностью (10, 50, 100). Продемонстрируйте качество их работы на примере поиска похожих слов (выберите любые 3 примера, соответствующих тематике корпуса из пункта 4)

In [99]:
ft_model_10 = FastText(
    sentences=cooking_sentences,
    vector_size=10,
    window=3,
    min_count=1,
    workers=2
)

In [100]:
ft_model_50 = FastText(
    sentences=cooking_sentences,
    vector_size=50,
    window=3,
    min_count=1,
    workers=2
)

In [101]:
ft_model_100 = FastText(
    sentences=cooking_sentences,
    vector_size=100,
    window=3,
    min_count=1,
    workers=2
)

In [109]:
def compare_models_ft(word):
    """Сравнивает представления слова в разных моделях"""
    print(f"\nСравнение для слова: '{word}'")

    # размерность 10
    try:
        w2v_similar = ft_model_10.wv.most_similar(word, topn=2)
        print(f"  FastText_10: {[w for w, _ in w2v_similar]}")
    except KeyError:
        print(f"  FastText_10: слово не найдено")

    # размерность 50
    try:
        ft_similar = ft_model_50.wv.most_similar(word, topn=2)
        print(f"  FastText_50: {[w for w, _ in ft_similar]}")
    except KeyError:
        print(f"  FastText_50: слово не найдено")

    # размерность 100
    try:
        ft_similar = ft_model_100.wv.most_similar(word, topn=2)
        print(f"  FastText_100: {[w for w, _ in ft_similar]}")
    except KeyError:
        print(f"  FastText_100: слово не найдено")

# Сравниваем для разных слов
compare_models_ft('картофель')
compare_models_ft('толченка')
compare_models_ft('молоко')
compare_models_ft('млеко')
compare_models_ft('соль')
compare_models_ft('пересолила')


Сравнение для слова: 'картофель'
  FastText_10: ['тесто', 'жарить']
  FastText_50: ['десерт', 'кипятить']
  FastText_100: ['хлеб', 'резать']

Сравнение для слова: 'толченка'
  FastText_10: ['парить', 'жарить']
  FastText_50: ['гриль', 'пирог']
  FastText_100: ['яйца', 'огурцы']

Сравнение для слова: 'молоко'
  FastText_10: ['ингредиенты', 'яблоки']
  FastText_50: ['десерт', 'яблоки']
  FastText_100: ['специи', 'сковорода']

Сравнение для слова: 'млеко'
  FastText_10: ['десерт', 'духовка']
  FastText_50: ['начинка', 'тост']
  FastText_100: ['взбивать', 'начинка']

Сравнение для слова: 'соль'
  FastText_10: ['запекать', 'сливки']
  FastText_50: ['десерт', 'мясо']
  FastText_100: ['уголь', 'мариновать']

Сравнение для слова: 'пересолила'
  FastText_10: ['соус', 'масло']
  FastText_50: ['соус', 'жарить']
  FastText_100: ['завтрак', 'яичница']


Для теста было выбрано 3 пары слов: одно слово, которое есть в корпусе, одно — которого нет (в том числе менее обычные варианты слов, сибирское диалектное "толченка" и польская калька "млеко")

Обучение на размерности 10 для "толченки" и "картофеля" дало наиболее положительный результат — совпал глагол "жарить". Следущие тесты также показывают, что размерность 10 дает хорошие результаты, более похожие на правду, однако слишком маленький размер корпуса не позволяет получить отличные результаты.

При увеличении размерности до 50 результат не становится хуже, однако лучше его можно назвать только для слов "соль"/"пересолила". Эти связи кажутся наименее случайными. Однако повторение одного и того же слова ("соус") для двух слов, одного из которых не было в корпусе, говорит о схожих результатах работы модели при размерности 50 и 10.

При размерности 100 модель выдает наиболее случайные слова, наименее связанные со словом. Например, "уголь" (вообще с едой слабо связано) или "сковорода" для молока (нелогично).

**Вывод**: чем меньше корпус, тем меньше нужна размерность.